# Segmental-duplication metrics across 4 species (Supplemental Figure S22)

Summarizes BISER segmental-duplication output for 4 species (human, mouse,
guinea pig, degu) as a 5-panel figure per species:

1. Number of segdup links (count)
2. Total merged alignment length (Mbp)
3. Distribution of segdup lengths (boxplot)
4. Average segdup length
5. Percent alignment error

Each species is plotted as its own 1×5 row and saved to
`segdup_metrics_<species>.png`. To add/remove species, edit `SPECIES` below.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")


In [ ]:
# BISER output dir + per-species config (reciprocal-duplicate-removed BEDPE).
BISER_DIR = f"{PROJ_ROOT}/code/command-line-script/genome-annotation/biser"

SPECIES = {
    "human": {
        "label": "Homo sapiens (Human)",
        "bedpe": f"{BISER_DIR}/human/segdup_output_human_duplicateLinkRemoved.bedpe",
    },
    "mouse": {
        "label": "Mus musculus (Mouse)",
        "bedpe": f"{BISER_DIR}/house_mouse/segdup_output_house_mouse_duplicateLinkRemoved.bedpe",
    },
    "guinea_pig": {
        "label": "Cavia porcellus (Guinea pig)",
        "bedpe": f"{BISER_DIR}/domesticated_guinea_pig/segdup_output_domesticated_guinea_pig_duplicateLinkRemoved.bedpe",
    },
    "degu": {
        "label": "Octodon degus (Degu)",
        "bedpe": f"{BISER_DIR}/hifiasm-041425/segdup_output_duplicateLinkRemoved.bedpe",
    },
}

BEDPE_COLS = ["chr1", "start1", "end1", "chr2", "start2", "end2", "reference",
              "score", "strand1", "strand2", "max_len", "aln_len", "cigar", "optional"]
CATEGORY_ORDER = ["Intrachromosomal", "Interchromosomal"]
COLORS = ["#87CEEB", "#F08080"]  # skyblue, lightcoral


In [ ]:
def merge_alignments(df, chrom_col="chr1", start_col="start1", end_col="end1", type_col="type"):
    """Merge overlapping/adjacent alignments of the same type per chromosome."""
    rows = []
    for (chrom, segdup_type), group in df.groupby([chrom_col, type_col]):
        sorted_group = group.sort_values(start_col)
        current_start = sorted_group.iloc[0][start_col]
        current_end = sorted_group.iloc[0][end_col]
        for _, r in sorted_group.iloc[1:].iterrows():
            s, e = r[start_col], r[end_col]
            if s <= current_end:
                current_end = max(current_end, e)
            else:
                rows.append({"chrom": chrom, "type": segdup_type, "start": current_start,
                             "end": current_end, "length": current_end - current_start})
                current_start, current_end = s, e
        rows.append({"chrom": chrom, "type": segdup_type, "start": current_start,
                     "end": current_end, "length": current_end - current_start})
    return pd.DataFrame(rows)


In [ ]:
def plot_species_summary(df, merged, label, out_png):
    """5-panel segdup-metric summary for one species (count, total length,
    length distribution, average length, percent alignment error)."""
    font_size = 15
    plt.figure(figsize=(22, 7))
    plt.suptitle(label, fontsize=20, fontweight="bold")

    type_counts = df["type"].value_counts().reindex(CATEGORY_ORDER)

    # 1) Count of segdup links
    plt.subplot(1, 5, 1)
    bars1 = plt.bar(range(len(type_counts)), type_counts.values, color=COLORS,
                    tick_label=type_counts.index)
    plt.title("Number of Segdup links", fontsize=font_size + 2)
    plt.ylabel("Count", fontsize=font_size + 2)
    plt.xlabel("Segdup Type", fontsize=font_size + 2)
    plt.xticks(fontsize=font_size - 1, rotation=10)
    plt.yticks(fontsize=font_size - 1)
    plt.ylim(0, max(type_counts.values) * 1.18)
    for i, (bar, count) in enumerate(zip(bars1, type_counts.values)):
        percentage = count / len(df) * 100
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01 * max(type_counts.values),
                 f"{count}\n({percentage:.1f}%)", ha="center", va="bottom",
                 fontweight="bold", fontsize=font_size)

    # 2) Total merged alignment length
    plt.subplot(1, 5, 2)
    aln_sum = merged.groupby("type")["length"].sum().reindex(CATEGORY_ORDER)
    aln_sum_mbp = aln_sum.values / 1e6
    bars5 = plt.bar(range(len(aln_sum)), aln_sum_mbp, color=COLORS, tick_label=aln_sum.index)
    plt.title("Total Merged Alignment Length", fontsize=font_size + 2)
    plt.ylabel("Total Merged Alignment Length (Mbp)", fontsize=font_size + 2)
    plt.xlabel("Segdup Type", fontsize=font_size + 2)
    plt.xticks(fontsize=font_size - 1, rotation=10)
    plt.yticks(fontsize=font_size - 1)
    plt.ylim(0, max(aln_sum_mbp) * 1.15)
    for i, (bar, val) in enumerate(zip(bars5, aln_sum_mbp)):
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01 * max(aln_sum_mbp),
                 f"{val:,.1f} Mbp", ha="center", va="bottom",
                 fontweight="bold", fontsize=font_size)

    # 3) Distribution of segdup lengths (boxplot)
    plt.subplot(1, 5, 3)
    sns.boxplot(data=df, x="type", y="avg_length", order=CATEGORY_ORDER, palette=COLORS)
    plt.title("Distribution of Segdup Lengths", fontsize=font_size + 2)
    plt.yscale("log")
    plt.ylabel("Average Length (bp)", fontsize=font_size + 2)
    plt.xlabel("Segdup Type", fontsize=font_size + 2)
    plt.xticks(fontsize=font_size - 1, rotation=10)
    plt.yticks(fontsize=font_size - 1)

    # 4) Average segdup length
    plt.subplot(1, 5, 4)
    means = df.groupby("type")["avg_length"].mean().reindex(CATEGORY_ORDER)
    bars3 = plt.bar(range(len(means)), means.values, color=COLORS, tick_label=means.index)
    plt.title("Average Segdup Length", fontsize=font_size + 2)
    plt.ylabel("Average Length (bp)", fontsize=font_size + 2)
    plt.xlabel("Segdup Type", fontsize=font_size + 2)
    plt.xticks(fontsize=font_size - 1, rotation=10)
    plt.yticks(fontsize=font_size - 1)
    plt.ylim(0, max(means.values) * 1.1)
    for i, (bar, mean_val) in enumerate(zip(bars3, means.values)):
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01 * max(means.values),
                 f"{mean_val:.0f} bp", ha="center", va="bottom",
                 fontweight="bold", fontsize=font_size)

    # 5) Percent alignment error (median of BISER 'score')
    plt.subplot(1, 5, 5)
    median = df.groupby("type")["score"].median().reindex(CATEGORY_ORDER)
    bars4 = plt.bar(range(len(median)), median.values, color=COLORS, tick_label=median.index)
    plt.title("Percent Alignment Error", fontsize=font_size + 2)
    plt.ylabel("Percent Alignment Error (%)", fontsize=font_size + 2)
    plt.xlabel("Segdup Type", fontsize=font_size + 2)
    plt.xticks(fontsize=font_size - 1, rotation=10)
    plt.yticks(fontsize=font_size - 1)
    plt.ylim(0, max(median.values) * 1.1)
    for i, (bar, median_val) in enumerate(zip(bars4, median.values)):
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01 * max(median.values),
                 f"{median_val:.0f} %", ha="center", va="bottom",
                 fontweight="bold", fontsize=font_size)

    plt.tight_layout()
    plt.subplots_adjust(top=0.87)
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


In [ ]:
# Run the summary for every species.
for species_key, cfg in SPECIES.items():
    print(f"=== {cfg['label']} ===")
    bedpe = pd.read_csv(cfg["bedpe"], sep="\t", header=None, names=BEDPE_COLS)
    df = bedpe.copy()

    # Intra- vs inter-chromosomal, and per-link lengths.
    df["type"] = df.apply(
        lambda r: "Intrachromosomal" if r["chr1"] == r["chr2"] else "Interchromosomal",
        axis=1,
    )
    df["length1"] = df["end1"] - df["start1"]
    df["length2"] = df["end2"] - df["start2"]
    df["avg_length"] = (df["length1"] + df["length2"]) / 2

    merged = pd.concat([
        merge_alignments(df[df["type"] == "Intrachromosomal"]),
        merge_alignments(df[df["type"] == "Interchromosomal"]),
    ])

    n_intra = int((df["type"] == "Intrachromosomal").sum())
    n_inter = int((df["type"] == "Interchromosomal").sum())
    print(f"  links: {len(df):,}  (intra {n_intra:,}, inter {n_inter:,})")

    plot_species_summary(df, merged, cfg["label"], f"segdup_metrics_{species_key}.png")
